# Physical point-probe retry v1 — NeZero repro first
CPU/high RAM, no GPU. Exactly one execution; no exploratory CI.
Repro → physical prerequisites → two point-probe draft audits. Not a cold seal;20/41 unchanged.


In [ ]:
import hashlib, pathlib, subprocess, sys, urllib.request, datetime, psutil
SOURCE_SHA = "91cc4dd5d6133e0eb0fc59279d58a71487caa6c7"
RUNNER_REV = "cmp99-point-probe-retry-v1"
LAUNCHER_URL = "https://raw.githubusercontent.com/lluiseriksson/THE-ERIKSSON-PROGRAMME/a2a33a3a9cde0bee572605b8d208ab8db7ad8037/scripts/launch_cmp99_point_probe_retry.py"
LAUNCHER_SHA256 = "73c556b63d431c62ea546fd38c3ecea13c4883351f19f2b2710e674d7eef107b"
assert psutil.virtual_memory().total / 2**30 >= 40, "HIGH_RAM_REQUIRED"
assert not pathlib.Path("/dev/nvidia0").exists(), "GPU_NOT_AUTHORIZED"
launcher = pathlib.Path("/content/launch_point_probe_retry_v1.py")
assert not launcher.exists(), "ALREADY_STARTED_NO_REEXECUTION"
with urllib.request.urlopen(LAUNCHER_URL, timeout=60) as response:
    payload = response.read()
assert hashlib.sha256(payload).hexdigest() == LAUNCHER_SHA256, "LAUNCHER_HASH_MISMATCH"
launcher.write_bytes(payload)
print("SOURCE_SHA=" + SOURCE_SHA + " RUNNER_REV=" + RUNNER_REV, flush=True)
print("HASH_GATE=PASS START_UTC=" + datetime.datetime.now(datetime.timezone.utc).isoformat(), flush=True)
with open("/content/point-probe-retry-v1-console.log", "xb") as log:
    child = subprocess.Popen([sys.executable, "-u", str(launcher)], stdout=log, stderr=subprocess.STDOUT)
    print("LAUNCH_PID=" + str(child.pid), flush=True)
    code = child.wait()
pathlib.Path("/content/point-probe-retry-v1-exit.txt").write_text(str(code) + "\n")
print(pathlib.Path("/content/point-probe-retry-v1-console.log").read_text(errors="replace")[-12000:], flush=True)
print("LAUNCHER_EXIT=" + str(code), flush=True)
if code:
    raise RuntimeError("Diagnostic stopped: preserve first error; do not reexecute")
